# COMP7103 Lab 12 · Catch the Fraud: Four Detectors, One Budget

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab12/Lab12_fraud_detectors.ipynb)

2,000 card transactions, each with an amount and a distance from the cardholder's home; 20 are confirmed fraud. Investigators can check **about 20 cases a day**. Four detectors rank the same transactions: which one should order their queue, and how far down the list should they go?

| Time | What you do |
|---|---|
| 5 min | Brief |
| 15 min | Guided work: score, then evaluate |
| 5 min | Share-out |

This lab is ungraded practice for the final exam. Data: synthetic card transactions made for this course (no real cardholders), CC0.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup (part of the 5-min brief)

This cell loads the transactions. Columns: `amount_hkd` (amount in HK$), `distance_km` (distance from home in km) and `is_fraud` (1 = confirmed fraud). Both features are skewed: many small values and a few huge ones. So every detector works on their logarithms, `X` = (log₁₀ amount, log₁₀ distance).

In [ ]:
import os, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.ensemble import IsolationForest
warnings.filterwarnings("ignore")

# A blank (___) that you have not filled in stops with: name '___' is not defined.
# (The next two lines make sure of that. You do not need to read or change them.)
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab12/data/"

# get(name) gives the path of a data file: the copy next to the notebook if there is one, otherwise a fresh
# download from the course GitHub. You do not need to read how it works.
def get(name):
    """Use the local file if it is there (Lab 12 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 12 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab12")
    return os.path.join("data", name)

df = pd.read_csv(get("transactions.csv"))
y = df["is_fraud"].values                                     # 1 = confirmed fraud
X = np.log10(df[["amount_hkd", "distance_km"]].values)        # the two features, on a log10 scale
N_FRAUD = int(y.sum())
print(len(df), "transactions ·", N_FRAUD, "confirmed fraud")

A first look. Grey rings: not fraud. Purple dots: the 20 frauds. There are two groups of normal transactions: a dense one (small local purchases) and a sparse one (travel and large purchases).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
normal = y == 0                          # True for the transactions that are not fraud
fraud = y == 1
ax.scatter(df["amount_hkd"][normal], df["distance_km"][normal], s=10, facecolors="none", edgecolors="grey",
           alpha=0.5, linewidths=0.6, label="not fraud")
ax.scatter(df["amount_hkd"][fraud], df["distance_km"][fraud], s=45, color="#6d4fc2", label="confirmed fraud")
ax.set_xscale("log")                     # log scales: the amounts range from a few dollars to tens of thousands
ax.set_yscale("log")
ax.set_xlabel("amount (HK$, log scale)")
ax.set_ylabel("distance from home (km, log scale)")
ax.set_title("2,000 card transactions")
ax.legend()
plt.show()
df[["amount_hkd", "distance_km"]].describe().round(1)     # count, mean, sd, min, quartiles, max of each column

The helpers (given). Every detector gives each transaction a score: the higher, the more unusual. `top_k()` returns the k highest-scoring transactions (the investigators' queue), `hits_at_k()` counts the frauds among them and `precision_at_k()` divides by k.

In [ ]:
def top_k(score, k):
    """Row numbers of the k transactions with the highest scores (rank 1 first)."""
    order = np.argsort(-score, kind="stable")      # row numbers sorted by score, highest first
    return order[:k]

def hits_at_k(score, k):
    """How many of the top k are confirmed fraud."""
    checked = top_k(score, k)
    return int(y[checked].sum())                   # y is 1 for fraud, so the sum counts the frauds

def precision_at_k(score, k):
    """Precision@k: the share of the k checked cases that are fraud."""
    return hits_at_k(score, k) / k

## 1 · Score every transaction (about 6 min)

**z-score (model-based).** The model: each feature is one bell-shaped (normal) cloud. For feature j with mean μⱼ and standard deviation (sd) σⱼ,

$$z_j = \frac{x_j - \mu_j}{\sigma_j}, \qquad \text{score}(x) = \sqrt{z_1^2 + z_2^2},$$

the distance from the mean in sd units. With one feature, the usual rule flags |z| > 3.

In [ ]:
Z = (X - X.mean(axis=0)) / ___                    # each feature's z-score: divide by its sd (one column per feature)
score_z = np.sqrt(___)                               # √(z₁² + z₂²), one number per transaction
flag_z = (np.abs(Z) > 3).any(axis=1)                 # the |z| > 3 rule on either feature (.any(axis=1): per row)
print(f"|z| > 3 rule: {flag_z.sum()} flagged, {y[flag_z].sum()} of them fraud")

**IQR rule (model-based).** Q₁ and Q₃ are the 25th and 75th percentiles of a feature, and IQR = Q₃ − Q₁ (the interquartile range). Flag x < Q₁ − 1.5 · IQR or x > Q₃ + 1.5 · IQR. Unlike the mean and sd, quartiles barely move when outliers are added. The rule is a yes/no flag, not a score.

In [ ]:
Q1 = np.percentile(X, 25, axis=0)      # the 25th percentile of each feature (axis=0: one value per column)
Q3 = np.percentile(X, 75, axis=0)      # the 75th percentile
IQR = Q3 - Q1
low = Q1 - ___ * IQR                        # the lower fence of each feature
high = Q3 + ___ * IQR                       # the upper fence
outside = (X < low) | (X > high)       # True where a value is outside its feature's fences (| means "or")
flag_iqr = outside.any(axis=1)         # outside the fences on either feature
print("fences, amount (HK$):  ", np.round(10 ** np.array([low[0], high[0]]), 1))
print("fences, distance (km): ", np.round(10 ** np.array([low[1], high[1]]), 1))
print(f"IQR rule: {flag_iqr.sum()} flagged, {y[flag_iqr].sum()} of them fraud")

Could the investigators check every transaction the IQR rule flags? It gives no order, so it cannot fill a queue of 20. From here on, the z-score stands for the model-based family.

**kNN distance (distance-based).** score(p) = d₅(p), the distance from p to its 5th nearest neighbour (kNN: k nearest neighbours; here we use 5 neighbours). We measure distances on the z-scores `Z`, so both features count equally.

In [ ]:
nn = NearestNeighbors(n_neighbors=5 + 1)          # + 1: each point is its own nearest neighbour (distance 0)
nn.fit(Z)
dist, positions = nn.kneighbors(Z)                 # dist: column 0 is the point itself, column j its j-th neighbour
score_knn = dist[:, ___]                           # d₅(p)

**LOF (density-based).** The local outlier factor compares a point's local density with its neighbours' densities. With Nₖ(p) the k nearest neighbours of p, d(p, o) the distance and dₖ(o) o's distance to its own k-th nearest neighbour,

$$\text{lrd}(p) = \frac{1}{\frac{1}{k}\sum_{o \in N_k(p)} \max\big(d_k(o),\, d(p, o)\big)}, \qquad \text{LOF}(p) = \frac{\frac{1}{k}\sum_{o \in N_k(p)} \text{lrd}(o)}{\text{lrd}(p)}.$$

lrd is the local reachability density. LOF ≈ 1: as dense as its neighbours; LOF ≫ 1: much sparser. We use LOF with 10 neighbours (k = 10 in the formula; not the k of the top-k list below). scikit-learn stores −LOF in `negative_outlier_factor_`, so we flip the sign.

**Isolation Forest.** Random trees cut a random feature at a random value until each point sits alone. Anomalies are isolated after few cuts. The score s(x) = 2^(−E[h(x)] / c(n)) comes from the average number of cuts h(x) over the trees (c(n) is the average path length for n points): close to 1 for anomalies, below 0.5 for normal points. scikit-learn's `score_samples` returns −s(x), so we flip the sign. We use 100 trees and a fixed seed.

The table under the next cell summarises the four scores. They are on different scales; only their order matters.

In [ ]:
lof = LocalOutlierFactor(n_neighbors=10)
lof.fit(Z)
score_lof = -lof.negative_outlier_factor_                    # LOF(p)
iso = IsolationForest(n_estimators=100, random_state=0)
iso.fit(Z)
score_if = -iso.score_samples(Z)                             # s(x)

SCORES = {"z-score": score_z, "kNN distance": score_knn, "LOF": score_lof, "Isolation Forest": score_if}
pd.DataFrame(SCORES).describe().round(2)                     # a summary of each score

## 2 · Evaluate: precision@k and recall@k (about 4 min)

Accuracy is useless here: flagging nothing is already 99% accurate. Instead, rank by score and check the top k:

$$\text{precision@}k = \frac{\text{frauds in the top } k}{k}, \qquad \text{recall@}k = \frac{\text{frauds in the top } k}{\text{all frauds } (20)}.$$

Fill in both, for k = 10, 20 and 50 and all four detectors.

In [ ]:
rows = {}
for name in SCORES:
    s = SCORES[name]                                   # this detector's scores
    row = {}
    for k in [10, 20, 50]:
        hits = hits_at_k(s, k)
        row[f"precision@{k}"] = hits / ___
        row[f"recall@{k}"] = hits / ___
    rows[name] = row
table = pd.DataFrame(rows).T.round(2)
table

Which detector finds the most fraud in the top 20, a day's work for the investigators?

In [ ]:
BEST = ___          # the name of the detector with the highest precision@20 (a string, or use .idxmax())
print("best at k = 20:", BEST)

## 3 · Explain the misses, then pick (about 5 min)

For each detector, the plot marks the 20 frauds: green if they are in its top 20, red if it missed them. Grey rings are normal transactions. Look at where the red crosses sit.

In [ ]:
def plot_misses(K):
    fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)   # 4 pictures, 2 x 2
    names = list(SCORES)
    normal = y == 0
    for i in range(4):
        ax = axes.ravel()[i]                 # the i-th picture
        name = names[i]
        caught = np.zeros(len(y), bool)      # True for the transactions in this detector's top K
        caught[top_k(SCORES[name], K)] = True
        hit = (y == 1) & caught              # frauds in the top K
        miss = (y == 1) & ~caught            # frauds not in the top K (~ means "not")
        ax.scatter(df["amount_hkd"][normal], df["distance_km"][normal], s=8, facecolors="none", edgecolors="grey",
                   alpha=0.4, linewidths=0.5)
        ax.scatter(df["amount_hkd"][hit], df["distance_km"][hit], s=60, color="green", label="fraud caught")
        ax.scatter(df["amount_hkd"][miss], df["distance_km"][miss], s=70, color="red", marker="X", label="fraud missed")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_title(f"{name}: {hit.sum()} caught, {miss.sum()} missed in its top {K}")
    for ax in axes[1]:                       # the bottom row
        ax.set_xlabel("amount (HK$, log scale)")
    for ax in axes[:, 0]:                    # the left column
        ax.set_ylabel("distance from home (km, log scale)")
    axes[0, 0].legend(loc="lower right")
    plt.tight_layout()
    plt.show()

plot_misses(K=20)

A check on the z-score's misses: on each feature alone, do they look normal? And where is the centre the z-score measures distance from?

In [ ]:
in_top20 = np.zeros(len(y), bool)                      # True for the z-score's top 20
in_top20[top_k(score_z, 20)] = True
missed_z = (y == 1) & ~in_top20                        # frauds outside the top 20 (~ means "not")
print("frauds the z-score misses in its top 20:", missed_z.sum())
print("  of these, |z| <= 3 on both features:     ", (missed_z & ~flag_z).sum())
print("  of these, inside both IQR fences:          ", (missed_z & ~flag_iqr).sum())
centre_amount = 10 ** X[:, 0].mean()                  # back from the log10 scale
centre_distance = 10 ** X[:, 1].mean()
print("centre the z-score measures from (mean of the log values): HK$", round(centre_amount), "·", round(centre_distance, 1), "km")

Answer in the cell below.

1. Look at the z-score panel. Where do most of the frauds the z-score misses sit: just outside the tight cluster, inside the sparse cluster, or far from both clusters? Which assumption of the z-score hid them?
2. Where do your best detector's misses sit?
3. Pick a detector and a k for investigators who check about 20 cases a day, with the two or three numbers that back it.

In [ ]:
WHERE_Z = "where the z-score's misses sit"
ASSUMPTION = "the assumption that hid them"
WHERE_BEST = "where the best detector's misses sit"
PICK = "detector"
K = 0
WHY = "two or three numbers from the table"
print(WHERE_Z, "|", ASSUMPTION, "|", WHERE_BEST, "|", PICK, "k =", K, "|", WHY)

## 4 · Share-out (5 min)

We collect these on the class board:

- **precision@20 for the z-score**, **precision@20 for the best detector**, and **recall@50 for the best detector**.
- Your vote: the **best detector at k = 20** (z-score, kNN distance, LOF or Isolation Forest).
- Your vote: **where the z-score's missed frauds sit** (just outside the tight cluster, inside the sparse cluster, or far from both clusters), and which assumption hid them.
- Your pick of a detector and a k for the investigators. A flagged case is a lead for a person to check, not a verdict.

The same precision@k reasoning answers the anomaly-evaluation question in Section C of the final exam.